# 08_3_3 DQA Scene Phase2 Anti-Drift Sweep

08_3_2ではDQA系のround1-2だけが伸び、round10まで回すと崩れました。
このノートブックは「早期改善を保存しつつphase2を続ける」ための5パターンを比較します。


In [ ]:
from __future__ import annotations

import json
import os
import signal
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())

DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_anti_drift_sweep.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_3_phase2_anti_drift_sweep"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_3_phase2_anti_drift_sweep"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_3_phase2_anti_drift_sweep"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT]:
    print(path, "exists=", path.exists())

BASE_WORK_ROOT.mkdir(parents=True, exist_ok=True)
BASE_STATS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)

## Variant Design

08_3_2の結果から、phase2の問題は「pseudoGTを使った更新が有効ではない」ではなく、「有効な序盤更新のあとも同じ強さで更新し続けて drift する」ことだと見ます。

- `a`: 08_3_2 best (`c`) をベースに、round後半でpseudo lossと集約残差を減衰
- `b`: `d`寄りの厳しめlocalization設計にpseudo数capを追加
- `c`: client更新をneck/headだけに制限して、backbone driftが原因か確認
- `d`: non_backbone更新でrecall補修を狙いつつ、bbox/clsを弱める
- `e`: 高precisionなphase1 round12をseedにして、少量DQAでrecallだけ戻せるか確認


In [ ]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 29780
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

# 空なら全 variant 実行。必要なら ["a_c_impulse_decay_r003"] のように絞る。
SELECTED_VARIANTS: list[str] = []

DEFAULT_POLICY = {
    "adapt_start_round": 2,
    "min_low": 0.39,
    "max_low": 0.50,
    "min_mid": 0.60,
    "max_mid": 0.78,
    "min_high": 0.86,
    "max_high": 0.96,
    "low_shift": 0.03,
    "high_shift": 0.06,
    "mid_gap": 0.22,
    "high_mid_gap": 0.18,
    "min_high_gap": 0.16,
    "max_nms": 0.48,
    "teacher_min": 0.22,
    "rare_count": 250.0,
    "rare_max_low": 0.44,
    "rare_max_mid": 0.68,
    "rare_max_high": 0.91,
    "low_step_limit": 0.018,
    "mid_step_limit": 0.022,
    "high_step_limit": 0.030,
    "nms_step_limit": 0.018,
    "low_mid_obj_weight": 0.30,
    "mid_high_obj_weight": 0.80,
}

VARIANTS = [
    {
        "name": "a_c_impulse_decay_r003",
        "description": "08_3_2 c bestを土台に、序盤だけpseudoGTを強く使い、後半はloss/residual/objを減衰して保存する。",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "client_lr0": 0.001,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.075,
        "server_anchor": 10.0,
        "temperature": 2.4,
        "stability_lambda": 0.55,
        "residual_start": 0.10,
        "residual_end": 0.005,
        "min_server_alpha_start": 0.74,
        "min_server_alpha_end": 0.92,
        "ssod": {
            "DQA0833_CLIENT_LR0_START": 0.0010,
            "DQA0833_CLIENT_LR0_END": 0.00025,
            "DQA0833_SERVER_LR0_START": 0.0030,
            "DQA0833_SERVER_LR0_END": 0.0010,
            "DQA0833_TEACHER_LOSS_WEIGHT_START": 0.34,
            "DQA0833_TEACHER_LOSS_WEIGHT_END": 0.20,
            "DQA0833_BOX_LOSS_WEIGHT_START": 0.010,
            "DQA0833_BOX_LOSS_WEIGHT_END": 0.000,
            "DQA0833_OBJ_LOSS_WEIGHT_START": 0.34,
            "DQA0833_OBJ_LOSS_WEIGHT_END": 0.18,
            "DQA0833_CLS_LOSS_WEIGHT_START": 0.040,
            "DQA0833_CLS_LOSS_WEIGHT_END": 0.000,
            "DQA0833_LOW_MID_OBJ_WEIGHT_START": 0.30,
            "DQA0833_LOW_MID_OBJ_WEIGHT_END": 0.10,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_START": 0.80,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_END": 0.25,
            "DQA0833_LOW_DELTA_START": 0.00,
            "DQA0833_LOW_DELTA_END": 0.035,
            "DQA0833_MID_DELTA_START": 0.00,
            "DQA0833_MID_DELTA_END": 0.055,
            "DQA0833_HIGH_DELTA_START": 0.00,
            "DQA0833_HIGH_DELTA_END": 0.060,
            "DQA0833_NMS_DELTA_START": 0.00,
            "DQA0833_NMS_DELTA_END": 0.040,
            "ET_MAX_PSEUDO_PER_IMAGE": 60,
            "ET_MAX_PSEUDO_PER_CLASS_IMAGE": 15,
            "DQA0833_IGNORE_OBJ": 0,
            "DQA0833_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0833_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {},
    },
    {
        "name": "b_strict_cap_decay_r003",
        "description": "08_3_2 dをさらに厳しくし、pseudo数capとbbox/cls早期減衰でlocalization崩壊を抑える。",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "client_lr0": 0.0008,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.050,
        "server_anchor": 14.0,
        "temperature": 2.8,
        "stability_lambda": 0.65,
        "residual_start": 0.075,
        "residual_end": 0.000,
        "min_server_alpha_start": 0.82,
        "min_server_alpha_end": 0.95,
        "ssod": {
            "DQA0833_CLIENT_LR0_START": 0.0008,
            "DQA0833_CLIENT_LR0_END": 0.00018,
            "DQA0833_SERVER_LR0_START": 0.0030,
            "DQA0833_SERVER_LR0_END": 0.0010,
            "DQA0833_TEACHER_LOSS_WEIGHT_START": 0.30,
            "DQA0833_TEACHER_LOSS_WEIGHT_END": 0.16,
            "DQA0833_BOX_LOSS_WEIGHT_START": 0.006,
            "DQA0833_BOX_LOSS_WEIGHT_END": 0.000,
            "DQA0833_OBJ_LOSS_WEIGHT_START": 0.30,
            "DQA0833_OBJ_LOSS_WEIGHT_END": 0.12,
            "DQA0833_CLS_LOSS_WEIGHT_START": 0.025,
            "DQA0833_CLS_LOSS_WEIGHT_END": 0.000,
            "DQA0833_LOW_MID_OBJ_WEIGHT_START": 0.20,
            "DQA0833_LOW_MID_OBJ_WEIGHT_END": 0.05,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_START": 0.65,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_END": 0.15,
            "DQA0833_LOW_DELTA_START": 0.02,
            "DQA0833_LOW_DELTA_END": 0.060,
            "DQA0833_MID_DELTA_START": 0.04,
            "DQA0833_MID_DELTA_END": 0.085,
            "DQA0833_HIGH_DELTA_START": 0.04,
            "DQA0833_HIGH_DELTA_END": 0.090,
            "DQA0833_NMS_DELTA_START": 0.02,
            "DQA0833_NMS_DELTA_END": 0.055,
            "ET_MAX_PSEUDO_PER_IMAGE": 40,
            "ET_MAX_PSEUDO_PER_CLASS_IMAGE": 8,
            "DQA0833_IGNORE_OBJ": 0,
            "DQA0833_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0833_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {
            "min_low": 0.42,
            "max_low": 0.52,
            "min_mid": 0.68,
            "max_mid": 0.82,
            "min_high": 0.91,
            "max_high": 0.98,
            "rare_max_low": 0.45,
            "rare_max_mid": 0.72,
            "rare_max_high": 0.94,
            "max_nms": 0.50,
        },
    },
    {
        "name": "c_neck_head_safe_r003",
        "description": "client更新をneck/headに限定し、backbone driftを切り分ける。pseudoGTは使うがglobalへの入り口は狭くする。",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "client_lr0": 0.0015,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.060,
        "server_anchor": 12.0,
        "temperature": 2.6,
        "stability_lambda": 0.60,
        "residual_start": 0.10,
        "residual_end": 0.015,
        "min_server_alpha_start": 0.78,
        "min_server_alpha_end": 0.92,
        "ssod": {
            "DQA0833_CLIENT_LR0_START": 0.0015,
            "DQA0833_CLIENT_LR0_END": 0.00035,
            "DQA0833_SERVER_LR0_START": 0.0030,
            "DQA0833_SERVER_LR0_END": 0.0010,
            "DQA0833_TEACHER_LOSS_WEIGHT_START": 0.32,
            "DQA0833_TEACHER_LOSS_WEIGHT_END": 0.18,
            "DQA0833_BOX_LOSS_WEIGHT_START": 0.008,
            "DQA0833_BOX_LOSS_WEIGHT_END": 0.001,
            "DQA0833_OBJ_LOSS_WEIGHT_START": 0.32,
            "DQA0833_OBJ_LOSS_WEIGHT_END": 0.16,
            "DQA0833_CLS_LOSS_WEIGHT_START": 0.030,
            "DQA0833_CLS_LOSS_WEIGHT_END": 0.003,
            "DQA0833_LOW_MID_OBJ_WEIGHT_START": 0.25,
            "DQA0833_LOW_MID_OBJ_WEIGHT_END": 0.08,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_START": 0.70,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_END": 0.22,
            "DQA0833_LOW_DELTA_START": 0.00,
            "DQA0833_LOW_DELTA_END": 0.040,
            "DQA0833_MID_DELTA_START": 0.00,
            "DQA0833_MID_DELTA_END": 0.060,
            "DQA0833_HIGH_DELTA_START": 0.01,
            "DQA0833_HIGH_DELTA_END": 0.070,
            "DQA0833_NMS_DELTA_START": 0.00,
            "DQA0833_NMS_DELTA_END": 0.040,
            "ET_MAX_PSEUDO_PER_IMAGE": 70,
            "ET_MAX_PSEUDO_PER_CLASS_IMAGE": 18,
            "DQA0833_IGNORE_OBJ": 0,
            "DQA0833_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0833_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {},
    },
    {
        "name": "d_non_backbone_recall_repair_r003",
        "description": "backboneを固定し、non_backboneでrecall補修。objは残し、bbox/clsはかなり弱くしてP崩れを抑える。",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "non_backbone",
        "client_lr0": 0.0012,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.050,
        "server_anchor": 13.0,
        "temperature": 2.7,
        "stability_lambda": 0.65,
        "residual_start": 0.085,
        "residual_end": 0.010,
        "min_server_alpha_start": 0.80,
        "min_server_alpha_end": 0.94,
        "ssod": {
            "DQA0833_CLIENT_LR0_START": 0.0012,
            "DQA0833_CLIENT_LR0_END": 0.00025,
            "DQA0833_SERVER_LR0_START": 0.0030,
            "DQA0833_SERVER_LR0_END": 0.0010,
            "DQA0833_TEACHER_LOSS_WEIGHT_START": 0.30,
            "DQA0833_TEACHER_LOSS_WEIGHT_END": 0.18,
            "DQA0833_BOX_LOSS_WEIGHT_START": 0.004,
            "DQA0833_BOX_LOSS_WEIGHT_END": 0.000,
            "DQA0833_OBJ_LOSS_WEIGHT_START": 0.34,
            "DQA0833_OBJ_LOSS_WEIGHT_END": 0.20,
            "DQA0833_CLS_LOSS_WEIGHT_START": 0.015,
            "DQA0833_CLS_LOSS_WEIGHT_END": 0.000,
            "DQA0833_LOW_MID_OBJ_WEIGHT_START": 0.28,
            "DQA0833_LOW_MID_OBJ_WEIGHT_END": 0.12,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_START": 0.85,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_END": 0.35,
            "DQA0833_LOW_DELTA_START": 0.00,
            "DQA0833_LOW_DELTA_END": 0.035,
            "DQA0833_MID_DELTA_START": 0.01,
            "DQA0833_MID_DELTA_END": 0.055,
            "DQA0833_HIGH_DELTA_START": 0.02,
            "DQA0833_HIGH_DELTA_END": 0.070,
            "DQA0833_NMS_DELTA_START": 0.00,
            "DQA0833_NMS_DELTA_END": 0.035,
            "ET_MAX_PSEUDO_PER_IMAGE": 80,
            "ET_MAX_PSEUDO_PER_CLASS_IMAGE": 20,
            "DQA0833_IGNORE_OBJ": 0,
            "DQA0833_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0833_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {},
    },
    {
        "name": "e_high_precision_seed_r012_recall_repair",
        "description": "phase1 round12の高precision seedから開始し、DQAで少量のrecall回復だけを狙う。",
        "source_phase1_round": 12,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "client_lr0": 0.0009,
        "server_lr0": 0.0025,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.050,
        "server_anchor": 15.0,
        "temperature": 3.0,
        "stability_lambda": 0.70,
        "residual_start": 0.080,
        "residual_end": 0.005,
        "min_server_alpha_start": 0.84,
        "min_server_alpha_end": 0.96,
        "ssod": {
            "DQA0833_CLIENT_LR0_START": 0.0009,
            "DQA0833_CLIENT_LR0_END": 0.00018,
            "DQA0833_SERVER_LR0_START": 0.0025,
            "DQA0833_SERVER_LR0_END": 0.0008,
            "DQA0833_TEACHER_LOSS_WEIGHT_START": 0.30,
            "DQA0833_TEACHER_LOSS_WEIGHT_END": 0.14,
            "DQA0833_BOX_LOSS_WEIGHT_START": 0.006,
            "DQA0833_BOX_LOSS_WEIGHT_END": 0.000,
            "DQA0833_OBJ_LOSS_WEIGHT_START": 0.30,
            "DQA0833_OBJ_LOSS_WEIGHT_END": 0.14,
            "DQA0833_CLS_LOSS_WEIGHT_START": 0.020,
            "DQA0833_CLS_LOSS_WEIGHT_END": 0.000,
            "DQA0833_LOW_MID_OBJ_WEIGHT_START": 0.20,
            "DQA0833_LOW_MID_OBJ_WEIGHT_END": 0.06,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_START": 0.60,
            "DQA0833_MID_HIGH_OBJ_WEIGHT_END": 0.18,
            "DQA0833_LOW_DELTA_START": 0.00,
            "DQA0833_LOW_DELTA_END": 0.045,
            "DQA0833_MID_DELTA_START": 0.02,
            "DQA0833_MID_DELTA_END": 0.070,
            "DQA0833_HIGH_DELTA_START": 0.03,
            "DQA0833_HIGH_DELTA_END": 0.085,
            "DQA0833_NMS_DELTA_START": 0.01,
            "DQA0833_NMS_DELTA_END": 0.045,
            "ET_MAX_PSEUDO_PER_IMAGE": 50,
            "ET_MAX_PSEUDO_PER_CLASS_IMAGE": 10,
            "DQA0833_IGNORE_OBJ": 0,
            "DQA0833_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0833_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {
            "min_low": 0.41,
            "max_low": 0.52,
            "min_mid": 0.66,
            "max_mid": 0.80,
            "min_high": 0.90,
            "max_high": 0.98,
            "rare_max_low": 0.44,
            "rare_max_mid": 0.70,
            "rare_max_high": 0.93,
            "max_nms": 0.50,
        },
    },
]

selected = [v for v in VARIANTS if not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS]
pd.DataFrame([
    {
        "name": v["name"],
        "seed_round": v["source_phase1_round"],
        "scope": v["client_train_scope"],
        "lr": f"{v['ssod'].get('DQA0833_CLIENT_LR0_START', v['client_lr0'])}->{v['ssod'].get('DQA0833_CLIENT_LR0_END', v['client_lr0'])}",
        "residual": f"{v['residual_start']}->{v['residual_end']}",
        "server_alpha": f"{v['min_server_alpha_start']}->{v['min_server_alpha_end']}",
        "description": v["description"],
    }
    for v in selected
])


In [ ]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def policy_value(variant: dict, key: str):
    return dict(DEFAULT_POLICY, **variant.get("policy", {}))[key]


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)

    env["DQA0833_VARIANT"] = variant["name"]
    env["DQA0833_PROFILE"] = variant["profile"]
    env["DQA0833_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0833_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0833_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0833_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0833_CLIENT_LR0"] = str(variant["client_lr0"])
    env["DQA0833_SERVER_LR0"] = str(variant["server_lr0"])
    env["DQA0833_CLIENT_ORTHOGONAL_WEIGHT"] = str(variant["client_orthogonal"])
    env["DQA0833_SERVER_ORTHOGONAL_WEIGHT"] = str(variant["server_orthogonal"])
    env["DQA0833_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0833_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0833_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0833_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0833_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_anti_drift_policy_schedule.jsonl").resolve())

    for key, value in variant.get("ssod", {}).items():
        env[key] = str(value)

    env["DQA08_ADAPT_START_ROUND"] = str(policy_value(variant, "adapt_start_round"))
    env["DQA08_MIN_LOW"] = str(policy_value(variant, "min_low"))
    env["DQA08_MAX_LOW"] = str(policy_value(variant, "max_low"))
    env["DQA08_MIN_MID"] = str(policy_value(variant, "min_mid"))
    env["DQA08_MAX_MID"] = str(policy_value(variant, "max_mid"))
    env["DQA08_MIN_HIGH"] = str(policy_value(variant, "min_high"))
    env["DQA08_MAX_HIGH"] = str(policy_value(variant, "max_high"))
    env["DQA08_LOW_SHIFT"] = str(policy_value(variant, "low_shift"))
    env["DQA08_HIGH_SHIFT"] = str(policy_value(variant, "high_shift"))
    env["DQA08_MID_GAP"] = str(policy_value(variant, "mid_gap"))
    env["DQA08_HIGH_MID_GAP"] = str(policy_value(variant, "high_mid_gap"))
    env["DQA08_MIN_HIGH_GAP"] = str(policy_value(variant, "min_high_gap"))
    env["DQA08_MAX_NMS"] = str(policy_value(variant, "max_nms"))
    env["DQA08_TEACHER_MIN"] = str(policy_value(variant, "teacher_min"))
    env["DQA08_RARE_COUNT"] = str(policy_value(variant, "rare_count"))
    env["DQA08_RARE_MAX_LOW"] = str(policy_value(variant, "rare_max_low"))
    env["DQA08_RARE_MAX_MID"] = str(policy_value(variant, "rare_max_mid"))
    env["DQA08_RARE_MAX_HIGH"] = str(policy_value(variant, "rare_max_high"))
    env["DQA08_LOW_STEP_LIMIT"] = str(policy_value(variant, "low_step_limit"))
    env["DQA08_MID_STEP_LIMIT"] = str(policy_value(variant, "mid_step_limit"))
    env["DQA08_HIGH_STEP_LIMIT"] = str(policy_value(variant, "high_step_limit"))
    env["DQA08_NMS_STEP_LIMIT"] = str(policy_value(variant, "nms_step_limit"))
    env.setdefault("DQA0833_LOW_MID_OBJ_WEIGHT", str(policy_value(variant, "low_mid_obj_weight")))
    env.setdefault("DQA0833_MID_HIGH_OBJ_WEIGHT", str(policy_value(variant, "mid_high_obj_weight")))
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root",
        str(variant_work_root(variant)),
        "--stats-root",
        str(variant_stats_root(variant)),
        "--phase1-rounds",
        "0",
        "--phase2-rounds",
        str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--gpus",
        str(GPUS),
        "--master-port",
        str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file",
        str(variant_train_log(variant)),
        "--classwise-blend",
        str(variant["classwise_blend"]),
        "--server-anchor",
        str(variant["server_anchor"]),
        "--temperature",
        str(variant["temperature"]),
        "--stability-lambda",
        str(variant["stability_lambda"]),
        "--dqa-start-phase",
        str(variant["dqa_start_phase"]),
        "--min-free-gib",
        str(MIN_FREE_GIB),
    ]
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "no pid"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "running?"
    return "running"


def tail_lines(path: Path, n: int = 30) -> list[str]:
    if not path.exists():
        return [f"missing: {path}"]
    return path.read_text(errors="replace").splitlines()[-n:]

print("helpers ready")

In [ ]:
# 実行セル。途中で止まっても history/global checkpoint があれば再利用します。
if not selected:
    raise RuntimeError("No selected variants")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    with runner_log.open("a", encoding="utf-8", buffering=1) as log:
        log.write("\n" + "=" * 100 + "\n")
        log.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
        if rc != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {rc}. See {runner_log}")
        print("variant completed:", variant["name"])


In [ ]:
# 進捗確認セル
rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append({
        "variant": variant["name"],
        "pid": read_pid(variant_pid_path(variant)),
        "pid_state": pid_state(read_pid(variant_pid_path(variant))),
        "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
        "latest": str(latest),
        "latest_exists": latest.exists(),
        "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
    })

display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    print("\n".join(tail_lines(variant_runner_log(variant), 20)))

In [ ]:
# final checkpoint + early checkpoint 評価。
# phase2 は early peak が出やすいので r01/r02/r03/r10 をまず見る。
EVAL_WORKSPACE = variant_work_root(selected[0])
REPORT_DIR = EVAL_WORKSPACE / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints: list[tuple[str, Path]] = []
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
old08_3_e02 = DQA_ROOT / "efficientteacher_dqa08_3_phase2_update_gating_sweep" / "e_ug_best_phase1_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0832_c01 = DQA_ROOT / "efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep" / "c_dqa_high_light_head_r003" / "global_checkpoints" / "phase2_round001_global.pt"
old0832_c02 = DQA_ROOT / "efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep" / "c_dqa_high_light_head_r003" / "global_checkpoints" / "phase2_round002_global.pt"
checkpoints.extend([
    ("p1_r003", seed03),
    ("p1_r012", seed12),
    ("old08_3_e_r02", old08_3_e02),
    ("old08_3_2_c_r001", old0832_c01),
    ("old08_3_2_c_r002", old0832_c02),
])

for variant in selected:
    root = variant_work_root(variant) / "global_checkpoints"
    for r in [1, 2, 3, PHASE2_ROUNDS_PER_VARIANT]:
        ckpt = root / f"phase2_round{r:03d}_global.pt"
        if ckpt.exists():
            checkpoints.append((f"{variant['name']}_r{r:03d}", ckpt))

print("eval_workspace:", EVAL_WORKSPACE)
print("checkpoints:")
for label, path in checkpoints:
    print(" ", label, path, "exists=", path.exists())

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    if path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0833_early_total.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)
else:
    print("No summary yet:", summary_csv)

In [ ]:
# 4 split final evaluation。上の early-total で良さそうな checkpoint をここに追加して比較。
BEST_LABELS = []  # 例: ["a_c_impulse_decay_r003_r002", "c_neck_head_safe_r003_r002"]

ckpt_map = {label: path for label, path in checkpoints if path.exists()}
if not BEST_LABELS:
    BEST_LABELS = [
        label for label in ckpt_map
        if label.endswith("_r001") or label.endswith("_r002") or label.endswith(f"r{PHASE2_ROUNDS_PER_VARIANT:03d}")
    ]

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "highway,citystreet,residential,total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label in ["p1_r003", "p1_r012", "old08_3_e_r02", "old08_3_2_c_r001", "old08_3_2_c_r002", *BEST_LABELS]:
    path = ckpt_map.get(label)
    if path and path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0833_selected_4splits.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)

In [ ]:
# pseudoGT / DQA gate 推移を確認するセル。
rows = []
for variant in selected:
    stats_root = variant_stats_root(variant)
    for r in range(1, PHASE2_ROUNDS_PER_VARIANT + 1):
        path = stats_root / f"phase2_round{r:03d}.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        counts = [0.0] * 10
        qsum = [0.0] * 10
        for client in data.get("clients", []):
            for i, value in enumerate(client.get("counts", [])):
                counts[i] += float(value)
            for i, value in enumerate(client.get("quality_sums", [])):
                qsum[i] += float(value)
        total = sum(counts)
        rows.append({
            "variant": variant["name"],
            "round": r,
            "pseudo_total": total,
            "mean_quality": sum(qsum) / total if total else None,
            "active_classes": sum(1 for x in counts if x > 0),
            "car_count": counts[2],
            "person_count": counts[0],
            "traffic_sign_count": counts[8],
        })

stats_df = pd.DataFrame(rows)
if not stats_df.empty:
    display(stats_df)
    display(stats_df.groupby("variant")[["pseudo_total", "mean_quality", "active_classes"]].agg(["min", "max", "mean"]))
else:
    print("No pseudo stats yet")

for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if not state_path.exists():
        continue
    state = json.loads(state_path.read_text())
    gate = state.get("phase2_anti_drift_sweep", [])
    if gate:
        print("\n", variant["name"])
        display(pd.DataFrame(gate))

## 期待する読み方

- まず `r001/r002` が 08_3_2 best を再現または上回るかを見る。
- 次に `r010` が `p1_r003` と `old08_3_2_c_r001/r002` からどれくらい落ちないかを見る。
- `c`/`d` が良ければ、full-parameter client drift が主因。
- `e` が良ければ、phase1を長く回した高precision seedからDQAでrecallだけ補う方向が有望。
- どれもr010で落ちるなら、phase2は「毎round更新」ではなく、quality gateを通ったroundだけglobalへ反映する設計に進む。
